# Action-conditioned fitted-return best response

This compares the existing flat Q head with a two-tower action-conditioned scorer. Both responders use identical complete-return data collection and are compared after equal measured training time against the exact best-response ceiling.

In [ ]:
from pathlib import Path
import gc
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy
from liars_poker.policies.neural import NeuralPolicy, compile_neural_to_dense
from liars_poker.policies.neural_q import compile_neural_q_to_dense
from liars_poker.policies.action_conditioned import (
    ActionFeatureEncoder,
    compile_action_conditioned_q_to_dense,
)
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.br_fitted_return import FittedReturnBRTrainer
from liars_poker.algo.br_fitted_return_action_conditioned import (
    ActionConditionedFittedReturnBRTrainer,
)
from liars_poker.eval.match_dense import evaluate_dense_response

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

reference_root = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs'
candidates = sorted(
    (path for path in reference_root.glob(f'{spec.to_short_str()}___*')
     if (path / 'policy' / 'metadata.json').exists()),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError('No completed Deep CFR reference policy found.')

REFERENCE_RUN_DIR = candidates[0]
target, target_spec = load_policy(str(REFERENCE_RUN_DIR / 'policy'))
assert target_spec == spec and isinstance(target, NeuralPolicy)
target.model_p1.to(device)
target.model_p2.to(device)
target.device = torch.device(device)
target.eval()

target_dense = compile_neural_to_dense(target, batch_size=65_536)
_, exact_meta = best_response_dense(spec, target_dense, store_state_values=False)
exact_first, exact_second = exact_meta['computer'].exploitability()
exact_exploitability = exact_first + exact_second - 1.0

print('reference:', REFERENCE_RUN_DIR)
print('claims:', len(target.rules.claims))
print('exact exploitability:', exact_exploitability)

In [ ]:
action_encoder = ActionFeatureEncoder(spec)
action_feature_table = pd.DataFrame(
    action_encoder.features,
    index=['CALL'] + [target.rules.render_action(i) for i in range(len(target.rules.claims))],
    columns=action_encoder.feature_names,
)
print('state input dimension:', spec.ranks + len(target.rules.claims))
print('action feature dimension:', action_encoder.feature_dim)
action_feature_table.head()

In [ ]:
SEEDS = (7, 17, 27)
BUDGETS_S = (0, 15, 30, 60, 120, 180, 300)
EPISODES_PER_ROLE = 512
ROLLOUT_BATCH_SIZE = 256

COMMON_KWARGS = dict(
    device=device,
    replay_capacity=1_000_000,
    batch_size=4096,
    learning_rate=1e-3,
    train_steps=100,
    warmup_transitions=20_000,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_decisions=500_000,
    rollouts_per_action=1,
)

VARIANTS = {
    'flat 512x512': (
        FittedReturnBRTrainer,
        dict(hidden_sizes=(512, 512)),
        compile_neural_q_to_dense,
    ),
    'action-conditioned': (
        ActionConditionedFittedReturnBRTrainer,
        dict(
            state_hidden_sizes=(512, 512),
            action_hidden_sizes=(128, 128),
            embedding_dim=256,
        ),
        compile_action_conditioned_q_to_dense,
    ),
}

def parameter_count(trainer):
    return sum(parameter.numel() for parameter in trainer.q_nets[0].parameters())

def exact_value(trainer, compiler):
    responder = compiler(trainer.policy(), batch_size=65_536)
    p_first, p_second = evaluate_dense_response(spec, target_dense, responder)
    return p_first, p_second, p_first + p_second - 1.0

In [ ]:
frames = []
for variant, (trainer_cls, variant_kwargs, compiler) in VARIANTS.items():
    for seed in SEEDS:
        print(f'\n=== {variant}; seed={seed} ===')
        trainer = trainer_cls(
            spec,
            target,
            seed=seed,
            **COMMON_KWARGS,
            **variant_kwargs,
        )
        measured_s = 0.0
        budget_idx = 0
        best_first = -np.inf
        best_second = -np.inf
        last = None
        rows = []

        while budget_idx < len(BUDGETS_S):
            if measured_s >= BUDGETS_S[budget_idx]:
                p_first, p_second, current = exact_value(trainer, compiler)
                best_first = max(best_first, p_first)
                best_second = max(best_second, p_second)
                best = best_first + best_second - 1.0
                row = {
                    'variant': variant,
                    'seed': seed,
                    'requested budget s': BUDGETS_S[budget_idx],
                    'measured training s': measured_s,
                    'iteration': trainer.iteration,
                    'parameters per role': parameter_count(trainer),
                    'current exploitability': current,
                    'best discovered exploitability': best,
                    'oracle gap': exact_exploitability - best,
                }
                if last is not None:
                    row.update({
                        'mean loss': np.mean([role['loss'] for role in last['roles']]),
                        'targets seen': sum(role['replay_seen'] for role in last['roles']),
                        'collect s': sum(role['collect_s'] for role in last['roles']),
                        'fit s': sum(role['fit_s'] for role in last['roles']),
                    })
                rows.append(row)
                print(
                    f"train={measured_s:7.1f}s iter={trainer.iteration:4d} "
                    f"best={best:.6f} gap={exact_exploitability - best:.6f}"
                )
                budget_idx += 1
                continue

            start = time.perf_counter()
            last = trainer.run_iteration(
                episodes_per_role=EPISODES_PER_ROLE,
                rollout_batch_size=ROLLOUT_BATCH_SIZE,
            )
            measured_s += time.perf_counter() - start

        frames.append(pd.DataFrame(rows))
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results = pd.concat(frames, ignore_index=True)
results.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
colors = dict(zip(VARIANTS, plt.rcParams['axes.prop_cycle'].by_key()['color']))

for variant, variant_frame in results.groupby('variant', sort=False):
    grouped = variant_frame.groupby('requested budget s')
    x = grouped['measured training s'].mean()
    best = grouped['best discovered exploitability'].mean()
    best_std = grouped['best discovered exploitability'].std()
    gap = grouped['oracle gap'].mean().clip(lower=1e-10)
    gap_std = grouped['oracle gap'].std()
    positive = x > 0
    color = colors[variant]
    axes[0].plot(x[positive], best[positive], marker='o', label=variant, color=color)
    axes[0].fill_between(x[positive], (best-best_std)[positive], (best+best_std)[positive], alpha=0.15, color=color)
    axes[1].loglog(x[positive], gap[positive], marker='o', label=variant, color=color)
    axes[1].fill_between(x[positive], (gap-gap_std).clip(lower=1e-10)[positive], (gap+gap_std)[positive], alpha=0.15, color=color)
    axes[2].plot(x[positive], grouped['fit s'].mean()[positive], marker='o', label=f'{variant}: fit', color=color)
    axes[2].plot(x[positive], grouped['collect s'].mean()[positive], marker='x', linestyle='--', label=f'{variant}: collect', color=color)

axes[0].axhline(exact_exploitability, color='black', linestyle='--', label='exact ceiling')
axes[0].set_xscale('log')
axes[0].set_title('Best discovered exploitability')
axes[1].set_title('Exact oracle gap')
axes[2].set_xscale('log')
axes[2].set_title('Per-iteration cost')
for axis in axes:
    axis.set_xlabel('Measured training seconds')
    axis.grid(alpha=0.3, which='both')
    axis.legend()
plt.tight_layout()

final = results.sort_values('measured training s').groupby(['variant', 'seed']).tail(1)
final.groupby('variant').agg({
    'parameters per role': 'first',
    'best discovered exploitability': ['mean', 'std'],
    'oracle gap': ['mean', 'std'],
    'collect s': 'mean',
    'fit s': 'mean',
})